# 02 - Weather Silver
### Goal
- import reusable code from src/
- use df.transform()
- apply UDF for wind classification
- write silver Delta table

In [0]:
%run ../01_setup/00_config

In [0]:
from transforms.weather_cleaning import standardize_weather_columns, filter_invalid_weather_rows
from transforms.business_rules import add_weather_flags
from udfs.weather_udfs import classify_wind_class

In [0]:
bronze_df = spark.table(bronze_weather_table)

silver_df = (
    bronze_df
    .transform(standardize_weather_columns)
    .transform(lambda df: filter_invalid_weather_rows(df, rules))
    .transform(lambda df: add_weather_flags(df, rules))
    .withColumn("wind_class", classify_wind_class(F.col("wind_speed_kmh")))
    .drop("_rescued_data")
)

display(silver_df.limit(10))
print(f"Bronze rows: {bronze_df.count()} - Silver rows: {silver_df.count()}")

In [0]:
silver_df.write.mode("overwrite").saveAsTable(silver_weather_table)
print(f"Silver table saved: {silver_weather_table}")